In [7]:
from pathlib import Path

# 根目录
DATA_ROOT = Path(r"D:\Study Abroad\course\DSCI498\Project\data\unpacked")

# 目标目录
target_dir = "6909"

target_path = DATA_ROOT / target_dir
parsed_path = target_path / "parsed"

print("Target folder:", target_path)

# ---------------------------------------------------
# locate audio
# ---------------------------------------------------

audio_files = list(target_path.glob("*.mp3")) + list(target_path.glob("*.ogg"))

if len(audio_files) == 0:
    raise Exception("No audio file found")

audio_path = audio_files[0]

# ---------------------------------------------------
# locate parsed files
# ---------------------------------------------------

notes_files = list(parsed_path.glob("*.notes.json"))
timing_files = list(parsed_path.glob("*.timing.json"))
metadata_files = list(parsed_path.glob("*.metadata.json"))

if len(notes_files) == 0:
    raise Exception("No notes.json found")

if len(timing_files) == 0:
    raise Exception("No timing.json found")

if len(metadata_files) == 0:
    raise Exception("No metadata.json found")

# 这里我们先选第一个 chart 做测试
notes_path = notes_files[0]
timing_path = timing_files[0]
metadata_path = metadata_files[0]

# ---------------------------------------------------
# print results
# ---------------------------------------------------

print("\nFiles located:")
print("audio_path   =", audio_path)
print("notes_path   =", notes_path)
print("timing_path  =", timing_path)
print("metadata_path=", metadata_path)

Target folder: D:\Study Abroad\course\DSCI498\Project\data\unpacked\6909

Files located:
audio_path   = D:\Study Abroad\course\DSCI498\Project\data\unpacked\6909\Lucky Star- Patty's (Kirby's sand canyon).mp3
notes_path   = D:\Study Abroad\course\DSCI498\Project\data\unpacked\6909\parsed\Nico Nico Douga - Patty's NIPPONPON (vytalibus) [Rokodo_[RL]'s Taiko Oni].notes.json
timing_path  = D:\Study Abroad\course\DSCI498\Project\data\unpacked\6909\parsed\Nico Nico Douga - Patty's NIPPONPON (vytalibus) [Rokodo_[RL]'s Taiko Oni].timing.json
metadata_path= D:\Study Abroad\course\DSCI498\Project\data\unpacked\6909\parsed\Nico Nico Douga - Patty's NIPPONPON (vytalibus) [Rokodo_[RL]'s Taiko Oni].metadata.json


In [8]:
import json
import librosa


# ---------------------------------------------------
# parse timing.json
# ---------------------------------------------------

with open(timing_path, "r", encoding="utf-8") as f:
    timing_data = json.load(f)

timing_points = timing_data.get("timing_points", [])

bpm_points = [
    tp for tp in timing_points
    if int(tp.get("uninherited", 0)) == 1
]

if len(bpm_points) == 0:
    raise Exception("No BPM timing points found")

bpm_points = sorted(bpm_points, key=lambda x: float(x["offset"]))

unique_ms_per_beat = sorted({
    round(float(tp["ms_per_beat"]), 10)
    for tp in bpm_points
})

if len(unique_ms_per_beat) != 1:
    raise Exception(f"Non-constant BPM detected: {unique_ms_per_beat}")

offset_ms = float(bpm_points[0]["offset"])
beat_duration_ms = float(bpm_points[0]["ms_per_beat"])
meter = int(bpm_points[0].get("meter", 4))
bpm = 60000.0 / beat_duration_ms
n_bpm_points = len(bpm_points)


# ---------------------------------------------------
# read audio info
# ---------------------------------------------------

y, sr = librosa.load(audio_path, sr=None, mono=True)

n_samples = len(y)
audio_duration_sec = n_samples / sr
audio_duration_ms = audio_duration_sec * 1000.0


# ---------------------------------------------------
# print result
# ---------------------------------------------------

print("Timing info:")
print("offset_ms       =", offset_ms)
print("beat_duration_ms=", beat_duration_ms)
print("bpm             =", bpm)
print("meter           =", meter)
print("n_bpm_points    =", n_bpm_points)

print("\nAudio info:")
print("sample_rate     =", sr)
print("n_samples       =", n_samples)
print("audio_duration_sec =", audio_duration_sec)
print("audio_duration_ms  =", audio_duration_ms)

Timing info:
offset_ms       = 470.0
beat_duration_ms= 378.489197287494
bpm             = 158.52500000000003
meter           = 4
n_bpm_points    = 1

Audio info:
sample_rate     = 44100
n_samples       = 3211776
audio_duration_sec = 72.82938775510205
audio_duration_ms  = 72829.38775510204


In [9]:
import numpy as np

# ---------------------------------------------------
# build beat grid
# ---------------------------------------------------

beat_times_ms = []
t = offset_ms

while t < audio_duration_ms:
    beat_times_ms.append(t)
    t += beat_duration_ms

beat_times_ms = np.array(beat_times_ms, dtype=float)
total_beats = len(beat_times_ms)

# 每个 beat 48 frames
frames_per_beat = 48
total_frames = total_beats * frames_per_beat

# ---------------------------------------------------
# print result
# ---------------------------------------------------

print("Beat grid info:")
print("total_beats  =", total_beats)
print("total_frames =", total_frames)

print("\nFirst 10 beat times (ms):")
print(beat_times_ms[:10])

print("\nLast 10 beat times (ms):")
print(beat_times_ms[-10:])

print("\nLast beat check:")
print("last_beat_time_ms =", beat_times_ms[-1])
print("audio_duration_ms =", audio_duration_ms)
print("remaining_tail_ms =", audio_duration_ms - beat_times_ms[-1])

Beat grid info:
total_beats  = 192
total_frames = 9216

First 10 beat times (ms):
[ 470.          848.48919729 1226.97839457 1605.46759186 1983.95678915
 2362.44598644 2740.93518372 3119.42438101 3497.9135783  3876.40277559]

Last 10 beat times (ms):
[69355.03390632 69733.52310361 70112.0123009  70490.50149819
 70868.99069547 71247.47989276 71625.96909005 72004.45828734
 72382.94748462 72761.43668191]

Last beat check:
last_beat_time_ms = 72761.43668191158
audio_duration_ms = 72829.38775510204
remaining_tail_ms = 67.95107319045928


In [10]:
import json
from collections import Counter

# ---------------------------------------------------
# load notes.json
# ---------------------------------------------------

with open(notes_path, "r", encoding="utf-8") as f:
    notes_data = json.load(f)

# 尝试自动找到事件列表
if isinstance(notes_data, list):
    events = notes_data
elif isinstance(notes_data, dict):
    if "notes" in notes_data:
        events = notes_data["notes"]
    elif "events" in notes_data:
        events = notes_data["events"]
    elif "hit_objects" in notes_data:
        events = notes_data["hit_objects"]
    else:
        raise Exception(f"Unknown notes.json structure. Top-level keys: {list(notes_data.keys())}")
else:
    raise Exception("notes.json has unsupported structure")

# ---------------------------------------------------
# basic summary
# ---------------------------------------------------

event_types = [e.get("type", "UNKNOWN") for e in events]
type_counter = Counter(event_types)

print("Notes info:")
print("total_events =", len(events))

print("\nEvent type counts:")
for k, v in sorted(type_counter.items()):
    print(f"{k}: {v}")

print("\nFirst 10 events:")
for i, e in enumerate(events[:10]):
    print(f"[{i}] {e}")

Notes info:
total_events = 413

Event type counts:
bigdon: 6
bigkat: 4
bpmchange: 1
don: 208
drumroll: 2
kat: 182
sliderend: 6
sliderstart: 4

First 10 events:
[0] {'type': 'don', 'time': 469, 'raw_time': 469, 'sv': 221.93500000000003, 'volume': 35, 'bpm': None, 'meter': None}
[1] {'type': 'bpmchange', 'time': 470, 'raw_time': 470, 'sv': 221.93500000000003, 'volume': 35, 'bpm': 158.52500000000003, 'meter': 4}
[2] {'type': 'kat', 'time': 659.2445986437469, 'raw_time': 659, 'sv': 221.93500000000003, 'volume': 35, 'bpm': None, 'meter': None}
[3] {'type': 'kat', 'time': 753.8668979656205, 'raw_time': 753, 'sv': 221.93500000000003, 'volume': 35, 'bpm': None, 'meter': None}
[4] {'type': 'don', 'time': 848.489197287494, 'raw_time': 848, 'sv': 221.93500000000003, 'volume': 35, 'bpm': None, 'meter': None}
[5] {'type': 'don', 'time': 941.0087788466592, 'raw_time': 942, 'sv': 221.93500000000003, 'volume': 35, 'bpm': None, 'meter': None}
[6] {'type': 'kat', 'time': 1037.733795931241, 'raw_time': 1

In [11]:
import pandas as pd

# ---------------------------------------------------
# convert events to dataframe
# ---------------------------------------------------

events_df = pd.DataFrame(events).copy()

# 只保留真正需要进入时序建模的事件
# 先保留全部，后面再决定是否排除 bpmchange
events_df["time"] = events_df["time"].astype(float)

# ---------------------------------------------------
# map to beat-relative coordinates
# ---------------------------------------------------

events_df["beat_position"] = (events_df["time"] - offset_ms) / beat_duration_ms
events_df["frame_position"] = events_df["beat_position"] * frames_per_beat
events_df["frame_index_rounded"] = events_df["frame_position"].round().astype(int)

# 额外看一下它落在哪个 beat、beat 内第几帧
events_df["beat_index"] = events_df["frame_index_rounded"] // frames_per_beat
events_df["frame_in_beat"] = events_df["frame_index_rounded"] % frames_per_beat

# ---------------------------------------------------
# print basic check
# ---------------------------------------------------

print("Mapped events preview:")
print(
    events_df[
        ["type", "time", "beat_position", "frame_position", "frame_index_rounded", "beat_index", "frame_in_beat"]
    ].head(20)
)

print("\nFrame range check:")
print("min frame_index_rounded =", events_df["frame_index_rounded"].min())
print("max frame_index_rounded =", events_df["frame_index_rounded"].max())
print("total_frames            =", total_frames)

print("\nEvents outside range:")
outside_df = events_df[
    (events_df["frame_index_rounded"] < 0) |
    (events_df["frame_index_rounded"] >= total_frames)
]
print("count =", len(outside_df))
if len(outside_df) > 0:
    print(outside_df[["type", "time", "frame_index_rounded"]].head(20))

Mapped events preview:
         type         time  beat_position  frame_position  \
0         don   469.000000      -0.002642       -0.126820   
1   bpmchange   470.000000       0.000000        0.000000   
2         kat   659.244599       0.500000       24.000000   
3         kat   753.866898       0.750000       36.000000   
4         don   848.489197       1.000000       48.000000   
5         don   941.008779       1.244444       59.733333   
6         kat  1037.733796       1.500000       72.000000   
7         don  1226.978395       2.000000       96.000000   
8         kat  1416.222993       2.500000      120.000000   
9         kat  1510.845293       2.750000      132.000000   
10        don  1605.467592       3.000000      144.000000   
11        kat  1794.712191       3.500000      168.000000   
12        don  1983.956789       4.000000      192.000000   
13        kat  2362.445986       5.000000      240.000000   
14        don  2740.935184       6.000000      288.000000   
1

In [12]:
# ---------------------------------------------------
# keep only modeling-related events
# ---------------------------------------------------

model_event_types = [
    "don", "kat", "bigdon", "bigkat",
    "drumroll", "sliderstart", "sliderend"
]

model_df = events_df[events_df["type"].isin(model_event_types)].copy()
model_df = model_df.sort_values(["frame_index_rounded", "time"]).reset_index(drop=True)

print("Model events info:")
print("total_model_events =", len(model_df))

print("\nEvent type counts:")
print(model_df["type"].value_counts().sort_index())

# ---------------------------------------------------
# inspect collisions: multiple events on same frame
# ---------------------------------------------------

frame_counts = model_df["frame_index_rounded"].value_counts().sort_index()
collision_frames = frame_counts[frame_counts > 1]

print("\nCollision check:")
print("number_of_collision_frames =", len(collision_frames))
print("total_events_in_collision_frames =", int(collision_frames.sum()) if len(collision_frames) > 0 else 0)

if len(collision_frames) > 0:
    print("\nFirst 20 collision frames:")
    print(collision_frames.head(20))

    collision_df = model_df[model_df["frame_index_rounded"].isin(collision_frames.index)].copy()
    print("\nCollision event details:")
    print(
        collision_df[
            ["type", "time", "frame_index_rounded", "beat_index", "frame_in_beat"]
        ].head(40)
    )
else:
    print("\nNo frame collisions found.")

Model events info:
total_model_events = 412

Event type counts:
type
bigdon           6
bigkat           4
don            208
drumroll         2
kat            182
sliderend        6
sliderstart      4
Name: count, dtype: int64

Collision check:
number_of_collision_frames = 0
total_events_in_collision_frames = 0

No frame collisions found.


In [13]:
import numpy as np

# ---------------------------------------------------
# build beat-aligned frame timeline
# ---------------------------------------------------

frame_duration_ms = beat_duration_ms / frames_per_beat

frame_times_ms = offset_ms + np.arange(total_frames) * frame_duration_ms

# 额外信息
sequence_beats = 4
frames_per_sequence = sequence_beats * frames_per_beat   # 4 * 48 = 192
total_sequences = total_frames // frames_per_sequence

# ---------------------------------------------------
# print result
# ---------------------------------------------------

print("Frame timeline info:")
print("frame_duration_ms   =", frame_duration_ms)
print("total_frames        =", total_frames)
print("frames_per_sequence =", frames_per_sequence)
print("total_sequences     =", total_sequences)

print("\nFirst 20 frame times (ms):")
print(frame_times_ms[:20])

print("\nFrames around beat boundary:")
print("frame 0   =", frame_times_ms[0])
print("frame 47  =", frame_times_ms[47])
print("frame 48  =", frame_times_ms[48])
print("frame 95  =", frame_times_ms[95])
print("frame 96  =", frame_times_ms[96])

print("\nLast 10 frame times (ms):")
print(frame_times_ms[-10:])

Frame timeline info:
frame_duration_ms   = 7.885191610156125
total_frames        = 9216
frames_per_sequence = 192
total_sequences     = 48

First 20 frame times (ms):
[470.         477.88519161 485.77038322 493.65557483 501.54076644
 509.42595805 517.31114966 525.19634127 533.08153288 540.96672449
 548.8519161  556.73710771 564.62229932 572.50749093 580.39268254
 588.27787415 596.16306576 604.04825737 611.93344898 619.81864059]

Frames around beat boundary:
frame 0   = 470.0
frame 47  = 840.6040056773379
frame 48  = 848.489197287494
frame 95  = 1219.0932029648318
frame 96  = 1226.978394574988

Last 10 frame times (ms):
[73061.0739631  73068.95915471 73076.84434632 73084.72953793
 73092.61472954 73100.49992115 73108.38511276 73116.27030437
 73124.15549598 73132.04068759]


In [15]:
import librosa
import numpy as np

# ---------------------------------------------------
# mel spectrogram parameters
# ---------------------------------------------------

n_fft = 2048
hop_length = 512
n_mels = 128

# ---------------------------------------------------
# build raw mel spectrogram
# ---------------------------------------------------

mel_spec = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_fft=n_fft,
    hop_length=hop_length,
    n_mels=n_mels,
    power=2.0
)

# 转成 log scale，更适合后续建模
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

# shape: (n_mels, n_frames) -> 转成 (n_frames, n_mels)
mel_spec_db = mel_spec_db.T

# ---------------------------------------------------
# original spectrogram frame times
# ---------------------------------------------------

orig_frame_times_sec = librosa.frames_to_time(
    np.arange(mel_spec_db.shape[0]),
    sr=sr,
    hop_length=hop_length,
    n_fft=n_fft
)
orig_frame_times_ms = orig_frame_times_sec * 1000.0

# ---------------------------------------------------
# print result
# ---------------------------------------------------

print("Raw mel spectrogram info:")
print("mel_spec_db shape =", mel_spec_db.shape)
print("n_orig_frames     =", mel_spec_db.shape[0])
print("n_mels            =", mel_spec_db.shape[1])

print("\nOriginal frame times (first 10 ms):")
print(orig_frame_times_ms[:10])

print("\nOriginal frame times (last 10 ms):")
print(orig_frame_times_ms[-10:])

print("\nAudio duration check:")
print("last_orig_frame_ms =", orig_frame_times_ms[-1])
print("audio_duration_ms  =", audio_duration_ms)

Raw mel spectrogram info:
mel_spec_db shape = (6274, 128)
n_orig_frames     = 6274
n_mels            = 128

Original frame times (first 10 ms):
[ 23.21995465  34.82993197  46.4399093   58.04988662  69.65986395
  81.26984127  92.87981859 104.48979592 116.09977324 127.70975057]

Original frame times (last 10 ms):
[72748.11791383 72759.72789116 72771.33786848 72782.9478458
 72794.55782313 72806.16780045 72817.77777778 72829.3877551
 72840.99773243 72852.60770975]

Audio duration check:
last_orig_frame_ms = 72852.60770975058
audio_duration_ms  = 72829.38775510204


In [16]:
import numpy as np

# ---------------------------------------------------
# interpolate raw mel spectrogram to beat-aligned frame timeline
# ---------------------------------------------------

aligned_mel_db = np.empty((len(frame_times_ms), mel_spec_db.shape[1]), dtype=np.float32)

for mel_bin in range(mel_spec_db.shape[1]):
    aligned_mel_db[:, mel_bin] = np.interp(
        frame_times_ms,
        orig_frame_times_ms,
        mel_spec_db[:, mel_bin],
        left=mel_spec_db[0, mel_bin],
        right=mel_spec_db[-1, mel_bin]
    )

# ---------------------------------------------------
# print result
# ---------------------------------------------------

print("Beat-aligned mel spectrogram info:")
print("aligned_mel_db shape =", aligned_mel_db.shape)

print("\nExpected shape:")
print("total_frames =", total_frames)
print("n_mels       =", mel_spec_db.shape[1])

print("\nFirst frame stats:")
print("min =", aligned_mel_db[0].min())
print("max =", aligned_mel_db[0].max())
print("mean =", aligned_mel_db[0].mean())

print("\nLast frame stats:")
print("min =", aligned_mel_db[-1].min())
print("max =", aligned_mel_db[-1].max())
print("mean =", aligned_mel_db[-1].mean())

print("\nAny NaN check:")
print(np.isnan(aligned_mel_db).any())

Beat-aligned mel spectrogram info:
aligned_mel_db shape = (9216, 128)

Expected shape:
total_frames = 9216
n_mels       = 128

First frame stats:
min = -80.0
max = -80.0
mean = -80.0

Last frame stats:
min = -80.0
max = -80.0
mean = -80.0

Any NaN check:
False


In [17]:
import numpy as np

# ---------------------------------------------------
# segment aligned mel into 4-beat sequences
# ---------------------------------------------------

audio_sequences = []

for seq_idx in range(total_sequences):
    start_frame = seq_idx * frames_per_sequence
    end_frame = start_frame + frames_per_sequence

    seq_mel = aligned_mel_db[start_frame:end_frame]

    if seq_mel.shape[0] != frames_per_sequence:
        raise Exception(f"Unexpected sequence length at seq_idx={seq_idx}: {seq_mel.shape}")

    audio_sequences.append(seq_mel)

audio_sequences = np.stack(audio_sequences, axis=0)

# ---------------------------------------------------
# print result
# ---------------------------------------------------

print("Audio sequence info:")
print("audio_sequences shape =", audio_sequences.shape)
print("expected shape        =", (total_sequences, frames_per_sequence, aligned_mel_db.shape[1]))

print("\nFirst sequence:")
print("shape =", audio_sequences[0].shape)
print("start_frame = 0")
print("end_frame   =", frames_per_sequence - 1)
print("start_time_ms =", frame_times_ms[0])
print("end_time_ms   =", frame_times_ms[frames_per_sequence - 1])

print("\nLast sequence:")
last_start = (total_sequences - 1) * frames_per_sequence
last_end = last_start + frames_per_sequence - 1
print("shape =", audio_sequences[-1].shape)
print("start_frame =", last_start)
print("end_frame   =", last_end)
print("start_time_ms =", frame_times_ms[last_start])
print("end_time_ms   =", frame_times_ms[last_end])

Audio sequence info:
audio_sequences shape = (48, 192, 128)
expected shape        = (48, 192, 128)

First sequence:
shape = (192, 128)
start_frame = 0
end_frame   = 191
start_time_ms = 470.0
end_time_ms   = 1976.0715975398198

Last sequence:
shape = (192, 128)
start_frame = 9024
end_frame   = 9215
start_time_ms = 71625.96909004886
end_time_ms   = 73132.04068758868


In [18]:
# ---------------------------------------------------
# build per-sequence event tokens
# ---------------------------------------------------

sequence_token_data = []

for seq_idx in range(total_sequences):
    seq_start_frame = seq_idx * frames_per_sequence
    seq_end_frame = seq_start_frame + frames_per_sequence - 1

    # 取出落在当前 sequence 内的事件
    seq_events = model_df[
        (model_df["frame_index_rounded"] >= seq_start_frame) &
        (model_df["frame_index_rounded"] <= seq_end_frame)
    ].copy()

    # 转成 sequence 内相对 frame
    seq_events["local_frame"] = seq_events["frame_index_rounded"] - seq_start_frame
    seq_events = seq_events.sort_values(["local_frame", "time"]).reset_index(drop=True)

    # 生成 token sequence
    tokens = []
    prev_local_frame = 0

    for _, row in seq_events.iterrows():
        local_frame = int(row["local_frame"])
        event_type = row["type"].upper()

        time_shift = local_frame - prev_local_frame
        if time_shift > 0:
            tokens.append(f"TS_{time_shift}")

        tokens.append(event_type)
        prev_local_frame = local_frame

    sequence_token_data.append({
        "seq_idx": seq_idx,
        "start_frame": seq_start_frame,
        "end_frame": seq_end_frame,
        "n_events": len(seq_events),
        "events_df": seq_events,
        "tokens": tokens
    })

# ---------------------------------------------------
# basic summary
# ---------------------------------------------------

print("Sequence token data built.")
print("total_sequences =", len(sequence_token_data))

event_counts = [x["n_events"] for x in sequence_token_data]
token_counts = [len(x["tokens"]) for x in sequence_token_data]

print("\nEvent count summary:")
print("min =", min(event_counts))
print("max =", max(event_counts))
print("mean =", sum(event_counts) / len(event_counts))

print("\nToken count summary:")
print("min =", min(token_counts))
print("max =", max(token_counts))
print("mean =", sum(token_counts) / len(token_counts))

# 看前3个 sequence
for i in range(3):
    print(f"\n=== Sequence {i} ===")
    print("frame range:", sequence_token_data[i]["start_frame"], "->", sequence_token_data[i]["end_frame"])
    print("n_events:", sequence_token_data[i]["n_events"])
    print("tokens:", sequence_token_data[i]["tokens"][:50])

Sequence token data built.
total_sequences = 48

Event count summary:
min = 1
max = 13
mean = 8.583333333333334

Token count summary:
min = 2
max = 25
mean = 16.1875

=== Sequence 0 ===
frame range: 0 -> 191
n_events: 11
tokens: ['DON', 'TS_24', 'KAT', 'TS_12', 'KAT', 'TS_12', 'DON', 'TS_12', 'DON', 'TS_12', 'KAT', 'TS_24', 'DON', 'TS_24', 'KAT', 'TS_12', 'KAT', 'TS_12', 'DON', 'TS_24', 'KAT']

=== Sequence 1 ===
frame range: 192 -> 383
n_events: 4
tokens: ['DON', 'TS_48', 'KAT', 'TS_48', 'DON', 'TS_48', 'KAT']

=== Sequence 2 ===
frame range: 384 -> 575
n_events: 9
tokens: ['DON', 'TS_24', 'DON', 'TS_12', 'DON', 'TS_12', 'KAT', 'TS_24', 'DON', 'TS_24', 'DON', 'TS_24', 'KAT', 'TS_24', 'KAT', 'TS_24', 'DON']


In [19]:
inspect_seq_idx = 0

seq_info = sequence_token_data[inspect_seq_idx]
seq_events = seq_info["events_df"].copy()
seq_tokens = seq_info["tokens"]

print(f"Sequence {inspect_seq_idx}")
print("frame range:", seq_info["start_frame"], "->", seq_info["end_frame"])
print("n_events:", seq_info["n_events"])

print("\nEvent table:")
print(seq_events[["type", "time", "frame_index_rounded", "local_frame"]])

print("\nTokens:")
print(seq_tokens)

Sequence 0
frame range: 0 -> 191
n_events: 11

Event table:
   type         time  frame_index_rounded  local_frame
0   don   469.000000                    0            0
1   kat   659.244599                   24           24
2   kat   753.866898                   36           36
3   don   848.489197                   48           48
4   don   941.008779                   60           60
5   kat  1037.733796                   72           72
6   don  1226.978395                   96           96
7   kat  1416.222993                  120          120
8   kat  1510.845293                  132          132
9   don  1605.467592                  144          144
10  kat  1794.712191                  168          168

Tokens:
['DON', 'TS_24', 'KAT', 'TS_12', 'KAT', 'TS_12', 'DON', 'TS_12', 'DON', 'TS_12', 'KAT', 'TS_24', 'DON', 'TS_24', 'KAT', 'TS_12', 'KAT', 'TS_12', 'DON', 'TS_24', 'KAT']


In [23]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import librosa


# =====================================================
# Config
# =====================================================

MAPPING_CSV = Path(r"D:\Study Abroad\course\DSCI498\Project\audio_chart_mapping.csv")
OUTPUT_CSV = Path(r"D:\Study Abroad\course\DSCI498\Project\data\unpacked\chart_preflight_summary_by_mapping.csv")

FRAMES_PER_BEAT = 48
SEQUENCE_BEATS = 4
FRAMES_PER_SEQUENCE = FRAMES_PER_BEAT * SEQUENCE_BEATS

MODEL_EVENT_TYPES = {
    "don", "kat", "bigdon", "bigkat",
    "drumroll", "sliderstart", "sliderend"
}

ALL_EXPECTED_TYPES = {
    "don", "kat", "bigdon", "bigkat",
    "drumroll", "sliderstart", "sliderend", "bpmchange"
}


# =====================================================
# Helpers
# =====================================================

def safe_path(x):
    if pd.isna(x):
        return None
    return Path(str(x))


def parse_timing_json(timing_path: Path):
    if timing_path is None or not timing_path.exists():
        raise FileNotFoundError(f"timing.json not found: {timing_path}")

    with open(timing_path, "r", encoding="utf-8") as f:
        timing_data = json.load(f)

    timing_points = timing_data.get("timing_points", [])
    if len(timing_points) == 0:
        raise ValueError("No timing_points found")

    bpm_points = [
        tp for tp in timing_points
        if int(tp.get("uninherited", 0)) == 1
    ]
    if len(bpm_points) == 0:
        raise ValueError("No BPM timing points found")

    bpm_points = sorted(bpm_points, key=lambda x: float(x["offset"]))

    unique_ms_per_beat = sorted({
        round(float(tp["ms_per_beat"]), 10)
        for tp in bpm_points
    })

    if len(unique_ms_per_beat) != 1:
        raise ValueError(f"Non-constant BPM detected: {unique_ms_per_beat}")

    offset_ms = float(bpm_points[0]["offset"])
    beat_duration_ms = float(bpm_points[0]["ms_per_beat"])
    bpm = 60000.0 / beat_duration_ms
    meter = int(bpm_points[0].get("meter", 4))

    return {
        "offset_ms": offset_ms,
        "beat_duration_ms": beat_duration_ms,
        "bpm": bpm,
        "meter": meter,
        "n_bpm_points": len(bpm_points),
        "n_timing_points": len(timing_points),
    }


def load_audio_info(audio_path: Path):
    if audio_path is None or not audio_path.exists():
        raise FileNotFoundError(f"audio file not found: {audio_path}")

    y, sr = librosa.load(audio_path, sr=None, mono=True)
    n_samples = len(y)
    audio_duration_sec = n_samples / sr
    audio_duration_ms = audio_duration_sec * 1000.0

    return {
        "sample_rate": sr,
        "n_samples": n_samples,
        "audio_duration_sec": audio_duration_sec,
        "audio_duration_ms": audio_duration_ms,
    }


def build_beat_grid_info(offset_ms: float, beat_duration_ms: float, audio_duration_ms: float):
    beat_times_ms = []
    t = offset_ms
    while t < audio_duration_ms:
        beat_times_ms.append(t)
        t += beat_duration_ms

    beat_times_ms = np.array(beat_times_ms, dtype=float)
    total_beats = len(beat_times_ms)
    total_frames = total_beats * FRAMES_PER_BEAT
    total_sequences = total_frames // FRAMES_PER_SEQUENCE

    frame_duration_ms = beat_duration_ms / FRAMES_PER_BEAT
    frame_times_ms = offset_ms + np.arange(total_frames) * frame_duration_ms

    return {
        "beat_times_ms": beat_times_ms,
        "frame_times_ms": frame_times_ms,
        "frame_duration_ms": frame_duration_ms,
        "total_beats": total_beats,
        "total_frames": total_frames,
        "total_sequences": total_sequences,
        "last_beat_time_ms": float(beat_times_ms[-1]) if total_beats > 0 else np.nan,
        "last_frame_time_ms": float(frame_times_ms[-1]) if total_frames > 0 else np.nan,
        "remaining_tail_ms": float(audio_duration_ms - beat_times_ms[-1]) if total_beats > 0 else np.nan,
        "frame_overshoot_ms": float(frame_times_ms[-1] - audio_duration_ms) if total_frames > 0 else np.nan,
    }


def load_notes(notes_path: Path):
    if notes_path is None or not notes_path.exists():
        raise FileNotFoundError(f"notes.json not found: {notes_path}")

    with open(notes_path, "r", encoding="utf-8") as f:
        notes_data = json.load(f)

    if isinstance(notes_data, list):
        events = notes_data
    elif isinstance(notes_data, dict):
        if "notes" in notes_data:
            events = notes_data["notes"]
        elif "events" in notes_data:
            events = notes_data["events"]
        elif "hit_objects" in notes_data:
            events = notes_data["hit_objects"]
        else:
            raise ValueError(f"Unknown notes.json structure. Keys: {list(notes_data.keys())}")
    else:
        raise ValueError("Unsupported notes.json structure")

    return events


def load_metadata(metadata_path: Path):
    if metadata_path is None or not metadata_path.exists():
        raise FileNotFoundError(f"metadata.json not found: {metadata_path}")

    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    return metadata


def analyze_events(events, offset_ms, beat_duration_ms, total_frames):
    events_df = pd.DataFrame(events).copy()

    if len(events_df) == 0:
        raise ValueError("No events found in notes.json")

    if "type" not in events_df.columns or "time" not in events_df.columns:
        raise ValueError(f"Required fields missing. Columns: {list(events_df.columns)}")

    events_df["time"] = events_df["time"].astype(float)

    event_counter = Counter(events_df["type"].tolist())
    unknown_types = sorted(set(event_counter.keys()) - ALL_EXPECTED_TYPES)

    events_df["beat_position"] = (events_df["time"] - offset_ms) / beat_duration_ms
    events_df["frame_position"] = events_df["beat_position"] * FRAMES_PER_BEAT
    events_df["frame_index_rounded"] = events_df["frame_position"].round().astype(int)

    model_df = events_df[events_df["type"].isin(MODEL_EVENT_TYPES)].copy()
    model_df = model_df.sort_values(["frame_index_rounded", "time"]).reset_index(drop=True)

    if len(model_df) > 0:
        min_model_frame = int(model_df["frame_index_rounded"].min())
        max_model_frame = int(model_df["frame_index_rounded"].max())
        first_model_time_ms = float(model_df["time"].min())
        last_model_time_ms = float(model_df["time"].max())
    else:
        min_model_frame = np.nan
        max_model_frame = np.nan
        first_model_time_ms = np.nan
        last_model_time_ms = np.nan

    outside_df = model_df[
        (model_df["frame_index_rounded"] < 0) |
        (model_df["frame_index_rounded"] >= total_frames)
    ]

    frame_counts = model_df["frame_index_rounded"].value_counts()
    collision_frames = frame_counts[frame_counts > 1]

    n_at_frame0 = int((model_df["frame_index_rounded"] == 0).sum()) if len(model_df) > 0 else 0
    n_at_last_frame = int((model_df["frame_index_rounded"] == total_frames - 1).sum()) if len(model_df) > 0 else 0

    return {
        "event_counter": event_counter,
        "unknown_types": unknown_types,
        "total_events": int(len(events_df)),
        "model_events": int(len(model_df)),
        "min_model_frame": min_model_frame,
        "max_model_frame": max_model_frame,
        "first_model_time_ms": first_model_time_ms,
        "last_model_time_ms": last_model_time_ms,
        "outside_event_count": int(len(outside_df)),
        "collision_frame_count": int(len(collision_frames)),
        "collision_event_total": int(collision_frames.sum()) if len(collision_frames) > 0 else 0,
        "n_at_frame0": n_at_frame0,
        "n_at_last_frame": n_at_last_frame,
    }


def summarize_one_mapping_row(row):
    record = {
        "row_id": row.name,
        "status": "ok",
        "error_message": "",
    }

    try:
        folder_id = row.get("folder_id", "")
        folder_path = row.get("folder_path", "")
        audio_file = row.get("audio_file", "")
        chart_base = row.get("chart_base", "")

        audio_path = safe_path(row.get("audio_path"))
        notes_path = safe_path(row.get("notes_path"))
        timing_path = safe_path(row.get("timing_path"))
        metadata_path = safe_path(row.get("metadata_path"))

        timing_info = parse_timing_json(timing_path)
        audio_info = load_audio_info(audio_path)
        grid_info = build_beat_grid_info(
            timing_info["offset_ms"],
            timing_info["beat_duration_ms"],
            audio_info["audio_duration_ms"]
        )
        events_info = analyze_events(
            load_notes(notes_path),
            timing_info["offset_ms"],
            timing_info["beat_duration_ms"],
            grid_info["total_frames"]
        )
        metadata = load_metadata(metadata_path)

        event_counter = events_info["event_counter"]

        record.update({
            # mapping identity
            "folder_id": folder_id,
            "folder_path": folder_path,
            "audio_file": audio_file,
            "chart_base": chart_base,

            # paths
            "audio_path": str(audio_path) if audio_path is not None else "",
            "notes_path": str(notes_path) if notes_path is not None else "",
            "timing_path": str(timing_path) if timing_path is not None else "",
            "metadata_path": str(metadata_path) if metadata_path is not None else "",

            # metadata
            "title": metadata.get("title", ""),
            "artist": metadata.get("artist", ""),
            "difficulty": metadata.get("difficulty", ""),
            "mode": metadata.get("mode", ""),
            "audio_filename_in_metadata": metadata.get("audio filename", metadata.get("audio_filename", "")),

            # timing
            "offset_ms": timing_info["offset_ms"],
            "beat_duration_ms": timing_info["beat_duration_ms"],
            "bpm": timing_info["bpm"],
            "meter": timing_info["meter"],
            "n_bpm_points": timing_info["n_bpm_points"],
            "n_timing_points": timing_info["n_timing_points"],

            # audio
            "sample_rate": audio_info["sample_rate"],
            "n_samples": audio_info["n_samples"],
            "audio_duration_sec": audio_info["audio_duration_sec"],
            "audio_duration_ms": audio_info["audio_duration_ms"],

            # grid
            "frame_duration_ms": grid_info["frame_duration_ms"],
            "total_beats": grid_info["total_beats"],
            "total_frames": grid_info["total_frames"],
            "total_sequences": grid_info["total_sequences"],
            "last_beat_time_ms": grid_info["last_beat_time_ms"],
            "last_frame_time_ms": grid_info["last_frame_time_ms"],
            "remaining_tail_ms": grid_info["remaining_tail_ms"],
            "frame_overshoot_ms": grid_info["frame_overshoot_ms"],

            # events
            "total_events": events_info["total_events"],
            "model_events": events_info["model_events"],
            "unknown_event_types": "|".join(events_info["unknown_types"]),
            "unknown_event_type_count": len(events_info["unknown_types"]),
            "min_model_frame": events_info["min_model_frame"],
            "max_model_frame": events_info["max_model_frame"],
            "first_model_time_ms": events_info["first_model_time_ms"],
            "last_model_time_ms": events_info["last_model_time_ms"],
            "outside_event_count": events_info["outside_event_count"],
            "collision_frame_count": events_info["collision_frame_count"],
            "collision_event_total": events_info["collision_event_total"],
            "n_at_frame0": events_info["n_at_frame0"],
            "n_at_last_frame": events_info["n_at_last_frame"],

            # event type counts
            "count_don": event_counter.get("don", 0),
            "count_kat": event_counter.get("kat", 0),
            "count_bigdon": event_counter.get("bigdon", 0),
            "count_bigkat": event_counter.get("bigkat", 0),
            "count_drumroll": event_counter.get("drumroll", 0),
            "count_sliderstart": event_counter.get("sliderstart", 0),
            "count_sliderend": event_counter.get("sliderend", 0),
            "count_bpmchange": event_counter.get("bpmchange", 0),

            # flags
            "flag_non4_meter": int(timing_info["meter"] != 4),
            "flag_no_model_events": int(events_info["model_events"] == 0),
            "flag_unknown_event_type": int(len(events_info["unknown_types"]) > 0),
            "flag_outside_events": int(events_info["outside_event_count"] > 0),
            "flag_collision": int(events_info["collision_frame_count"] > 0),
            "flag_negative_offset": int(timing_info["offset_ms"] < 0),
            "flag_last_frame_overshoot": int(grid_info["frame_overshoot_ms"] > 0),
            "flag_no_full_sequence": int(grid_info["total_sequences"] == 0),
        })

    except Exception as e:
        record["status"] = "error"
        record["error_message"] = str(e)

        # 把 mapping 里的基础信息也尽量保留下来，方便排查
        record["folder_id"] = row.get("folder_id", "")
        record["folder_path"] = row.get("folder_path", "")
        record["audio_file"] = row.get("audio_file", "")
        record["chart_base"] = row.get("chart_base", "")
        record["audio_path"] = str(row.get("audio_path", ""))
        record["notes_path"] = str(row.get("notes_path", ""))
        record["timing_path"] = str(row.get("timing_path", ""))
        record["metadata_path"] = str(row.get("metadata_path", ""))

    return record


# =====================================================
# Main
# =====================================================

def main():
    if not MAPPING_CSV.exists():
        raise FileNotFoundError(f"Mapping CSV not found: {MAPPING_CSV}")

    mapping_df = pd.read_csv(MAPPING_CSV)
    print("Loaded mapping rows:", len(mapping_df))

    required_cols = [
        "folder_id", "folder_path", "audio_file", "audio_path",
        "chart_base", "notes_path", "timing_path", "metadata_path"
    ]
    missing_cols = [c for c in required_cols if c not in mapping_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in mapping CSV: {missing_cols}")

    records = []
    total_rows = len(mapping_df)

    for i, (_, row) in enumerate(mapping_df.iterrows(), start=1):
        if i % 20 == 0 or i == 1 or i == total_rows:
            print(f"[{i}/{total_rows}] Processing folder_id={row.get('folder_id', '')} | chart={row.get('chart_base', '')}")
        rec = summarize_one_mapping_row(row)
        records.append(rec)

    df = pd.DataFrame(records)

    first_cols = [
        "row_id",
        "folder_id", "chart_base", "status", "error_message",
        "title", "artist", "difficulty", "mode",
        "bpm", "meter", "offset_ms", "beat_duration_ms",
        "audio_duration_ms", "total_beats", "total_frames", "total_sequences",
        "total_events", "model_events",
        "outside_event_count", "collision_frame_count", "collision_event_total",
        "flag_non4_meter", "flag_no_model_events", "flag_unknown_event_type",
        "flag_outside_events", "flag_collision", "flag_negative_offset",
        "flag_last_frame_overshoot", "flag_no_full_sequence",
        "audio_path", "notes_path", "timing_path", "metadata_path",
    ]
    remaining_cols = [c for c in df.columns if c not in first_cols]
    df = df[first_cols + remaining_cols]

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print("\nDone.")
    print("Saved to:")
    print(OUTPUT_CSV)

    print("\nStatus summary:")
    print(df["status"].value_counts(dropna=False))

    flag_cols = [c for c in df.columns if c.startswith("flag_")]
    print("\nFlag summary:")
    for col in flag_cols:
        print(f"{col}: {int(df[col].fillna(0).sum())}")

    print("\nRows with errors or flags (first 50):")
    issue_mask = (df["status"] != "ok")
    for col in flag_cols:
        issue_mask = issue_mask | (df[col].fillna(0) > 0)

    issue_cols = [
        "row_id", "folder_id", "chart_base", "status", "error_message",
        "bpm", "meter", "offset_ms",
        "total_events", "model_events",
        "outside_event_count", "collision_frame_count",
        "flag_non4_meter", "flag_no_model_events",
        "flag_unknown_event_type", "flag_outside_events",
        "flag_collision", "flag_negative_offset",
        "flag_last_frame_overshoot", "flag_no_full_sequence"
    ]
    print(df.loc[issue_mask, issue_cols].head(50))


if __name__ == "__main__":
    main()

Loaded mapping rows: 290
[1/290] Processing folder_id=1024653 | chart=Mizukara o Enshutsu Suru Otome no Kai - Girlish Lover (yuki_momoiro722) [Genjuro's Futsuu]
[20/290] Processing folder_id=1175317 | chart=Kendrick Lamar - King Kunta (Horiiizon) [Genjuro's Kantan]
[40/290] Processing folder_id=1249951 | chart=Kobaryo - New Game Plus (Genjuro) [Challenge]
[60/290] Processing folder_id=1292034 | chart=Chroma - Sayonara Planet Wars (Mid) [2199's Oni]
[80/290] Processing folder_id=1656103 | chart=Chroma - fly in the galaxy (ikin5050) [Muzukashii]
[100/290] Processing folder_id=1966609 | chart=Shizuru (CV Nabatame Hitomi), Rino (CV Asumi Kana) - SUPER CHOCOLATE (Game Ver.) ([-E S I A-]) [MUZUKASHII]
[120/290] Processing folder_id=2022269 | chart=MyGO!!!!! - Hitoshizuku (TV Size) (-Maruko-) [Jhindred's Oni]
[140/290] Processing folder_id=2117806 | chart=Nor - Usagi Flap (doulph) [Futsuu]
[160/290] Processing folder_id=2300928 | chart=Hamige - Bonfire (-Koppe-) [Inner Oni]
[180/290] Processi

C:\Users\Xiang Gao\AppData\Local\Temp\ipykernel_14548\3354034634.py:89: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=None, mono=True)
C:\Users\Xiang Gao\AppData\Roaming\Python\Python311\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


[220/290] Processing folder_id=315288 | chart=765PRO ALLSTARS - Jibun REST@RT (Cut Ver.) (KanaRin) [Inner Oni]
[240/290] Processing folder_id=591845 | chart=Doubutsu Biscuits x PPP - Youkoso JAPARI PARK e (TV size ver.) (butter0414) [Muzukashii]
[260/290] Processing folder_id=74301 | chart=MomoKuro-tei Ichimon - Nippon Egao Hyakkei (TV Size) (CXu) [Kinomi's Oni]
[280/290] Processing folder_id=914056 | chart=Petit Rabbit's - Happiness Encore (Tyistiana) [Smallwu's Oni]
[290/290] Processing folder_id=972634 | chart=Verdammt - Hitoribocchi no Maou (Axer) [komasy's Muzukashii]

Done.
Saved to:
D:\Study Abroad\course\DSCI498\Project\data\unpacked\chart_preflight_summary_by_mapping.csv

Status summary:
status
ok    290
Name: count, dtype: int64

Flag summary:
flag_non4_meter: 16
flag_no_model_events: 0
flag_unknown_event_type: 0
flag_outside_events: 6
flag_collision: 0
flag_negative_offset: 23
flag_last_frame_overshoot: 281
flag_no_full_sequence: 0

Rows with errors or flags (first 50):
    